In [ ]:
# @title
!pip install -q groq pandas

import time
import pandas as pd
from getpass import getpass
from IPython.display import display
from groq import Groq, AuthenticationError, RateLimitError, BadRequestError
from groq import Groq

GROQ_API_KEY = "API_KEY"
client = Groq(api_key=GROQ_API_KEY)

models = client.models.list()
for model in models.data:
    print(model.id)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.7 MB/s eta 0:00:00
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
whisper-large-v3
qwen/qwen3-32b
groq/compound
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
llama-3.1-8b-instant
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
allam-2-7b
whisper-large-v3-turbo
llama-3.3-70b-versatile


In [ ]:
# ============================================================================
# CELL 1 — Imports & Configuration
# ============================================================================
!pip install groq -q

import json, re, time, logging
import pandas as pd
from groq import Groq

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

GROQ_API_KEY = "API_KEY"
client       = Groq(api_key=GROQ_API_KEY)

GROQ_MODELS = [
    "meta-llama/llama-4-scout-17b-16e-instruct",
]

BASE_PATH   = "BASE_PATH"
INPUT_PATH  = f"{BASE_PATH}/Input_Data_GEC.xlsx"
OUTPUT_PATHS = {
    "meta-llama/llama-4-scout-17b-16e-instruct": f"{BASE_PATH}/Llama4_Scout_17B_Responses_Data_GEC.xlsx",
}

SYSTEM_PROMPT = """You are an expert in Odia (ଓଡ଼ିଆ) linguistics specializing in Grammatical Error Detection (GED) and Grammatical Error Correction (GEC). Analyze the given text and identify all linguistic errors with high precision.

## Categories (apply in this priority order — assign only the highest-priority matching category per span)

1. Script Normalization — Unicode-level encoding errors where the character sequence is wrong even if the rendering looks similar to the correct form. Includes: nukta misplacement relative to a base consonant, incorrect virama usage in conjunct formation, vowel sign decomposition errors where two combining marks are used instead of a single precomposed one, and ZWNJ/ZWJ presence or absence errors affecting ligature formation or consonant cluster boundaries. Also includes inappropriate consonant conjunct or cluster formation caused by the absence of a ZWNJ, where two adjacent consonants incorrectly merge into a ligature instead of remaining as distinct units.

2. Spelling & Typographical Errors — Unicode encoding is correct but the wrong characters are used. Includes: confusion between short and long vowel signs, substitution among phonetically similar consonants, missing or extra characters within a word, and incorrect word boundaries (two words merged or one word split). [*Note: if the error is in how characters are encoded rather than which characters are chosen, classify as Script Normalization instead.]

3. Grammatical Errors — Morphosyntactic errors where the script and spelling are correct. Includes: wrong verb tense, aspect, or inflectional form; agreement failure; incorrect postpositional case marker; wrong conjunction for the syntactic context; incorrect voice; light verb misuse; word order violations; wrong copular constructions; and missing punctuation that is grammatically required.

4. Code-Mixing / Wrong Language — Odia text containing elements from a different language or script. Includes: Roman-script words or abbreviations where an Odia equivalent exists, non-Odia numeral systems where Odia digits are standard, characters from another Indic script embedded in Odia text, and loanword phrases where a standard Odia term is available.

5. Correct Sentence / No Errors — Return this as a single entry if no errors are found.

## Rules
- Priority: If a span qualifies under multiple categories, assign only the highest-priority matching category (1 is highest).
- Single error assumption: Each input sentence contains exactly one error, which may span a single character, a word, or a longer phrase (e.g., a word order violation). Identify the single best error span and assign it to exactly one category. If multiple categories seem applicable, rank them by the priority order above and assign the highest-ranking one.
- Span overlap: Report the one erroneous span only. Do not report sub-spans of the same error separately.
- Correction scope: Make minimum necessary edits only — do not reorder, paraphrase, or add content beyond what fixing the identified error requires.
- Binary flag: Set "has_errors" to true if an error is found, and false if the sentence is correct. When false, "errors" must be an empty array and "corrected_sentence" must be identical to the input.

## Output
Return ONLY valid JSON. No markdown, no text outside the JSON.

{
  "has_errors": <true | false>,
  "errors": [
    {
      "error_span": "<exact erroneous text>",
      "category": "<one of the five categories>",
      "description": "<technical explanation>"
    }
  ],
  "corrected_sentence": "<fully corrected sentence, or identical to input if no errors>"
}"""

In [ ]:
# ============================================================================
# CELL 2 — API Call & Response Parsing
# ============================================================================
def parse_response(text: str):
    text = re.sub(r'^```json\s*|^```\s*|\s*```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        start, end = text.find('{'), text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
    except json.JSONDecodeError:
        pass
    return None


def call_groq(sentence: str, sentence_id: str, model_id: str) -> dict:
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": sentence},
            ],
            temperature=0.0,
            top_p=1.0,
            max_tokens=4096,
        )
        logger.info(f"[{sentence_id}][{model_id}] finish: {resp.choices[0].finish_reason}")

        content = resp.choices[0].message.content
        parsed  = parse_response(content) if content else None
        return {"status": "success", **parsed} if parsed else {"status": "parse_failed", "raw": content}

    except Exception as e:
        logger.exception(f"[{sentence_id}][{model_id}] Exception")
        return {"status": "exception", "error": str(e)}


def extract_fields(errors):
    if not errors:
        return "", "", ""
    spans        = " | ".join(str(e.get("error_span", ""))  for e in errors)
    categories   = " | ".join(str(e.get("category", ""))    for e in errors)
    descriptions = " | ".join(str(e.get("description", "")) for e in errors)
    return spans, categories, descriptions

In [ ]:
# ============================================================================
# CELL 3 — Run All Models & Save to Separate Excel Files (Resume-Safe)
# ============================================================================
import os
import re as _re

def _parse_retry_delay(error_str: str, default: float = 65.0) -> float:
    """Parse Groq retry delay — handles formats like 2m17.376s, 1m9s, 30s."""
    m = _re.search(r'(?:(\d+)m)?(\d+(?:\.\d+)?)s', error_str)
    if m:
        minutes = float(m.group(1)) if m.group(1) else 0.0
        seconds = float(m.group(2)) if m.group(2) else 0.0
        return minutes * 60 + seconds + 5.0
    return default

def _is_daily_limit(error_str: str) -> bool:
    return "per day" in error_str.lower() or "tpd" in error_str.lower()


df = pd.read_excel(INPUT_PATH, dtype=str).fillna("")
df = df[df["Incorrect Sentences"].str.strip() != ""].reset_index(drop=True)
logger.info(f"Loaded {len(df)} sentences")

failed_statuses = {"api_error", "parse_failed", "exception"}
all_records = {}

for model_id in GROQ_MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_id}")
    print(f"{'='*60}")

    # ── Load existing progress if file exists ──────────────────────────────
    output_path = OUTPUT_PATHS[model_id]
    if os.path.exists(output_path):
        existing_df = pd.read_excel(output_path, dtype=str).fillna("")
        records = existing_df.to_dict(orient="records")
        done_sids = {
            r["Sentence_ID"]
            for r in records
            if r.get("Status") not in failed_statuses and r.get("Status") != ""
        }
        logger.info(f"[{model_id}] Resuming — {len(done_sids)} already done, "
                    f"{len(df) - len(done_sids)} remaining.")
    else:
        records  = []
        done_sids = set()
        logger.info(f"[{model_id}] No existing file — starting fresh.")

    # Build a lookup so we can update failed rows in-place
    sid_to_idx = {r["Sentence_ID"]: i for i, r in enumerate(records)}

    skip_model = False

    for _, row in df.iterrows():
        if skip_model:
            break

        sid, sentence = row["Sentence_ID"], row["Incorrect Sentences"]

        # Skip if already successfully processed
        if sid in done_sids:
            continue

        for attempt in range(2):
            result = call_groq(sentence, sid, model_id)

            if result.get("status") == "exception":
                err = result.get("error", "")

                if "404" in err or "not found" in err.lower():
                    logger.warning(f"[{model_id}] Model not found. Skipping entire model.")
                    skip_model = True
                    break

                if "429" in err or "rate limit" in err.lower():
                    if _is_daily_limit(err):
                        logger.warning(f"[{model_id}] Daily token limit hit. Skipping model — retry tomorrow.")
                        skip_model = True
                        break
                    if attempt == 0:
                        wait = _parse_retry_delay(err)
                        logger.warning(f"[{sid}][{model_id}] Rate limited. Waiting {wait:.0f}s then retrying...")
                        time.sleep(wait)
                        continue

            break

        if skip_model:
            break

        errors = result.get("errors", [])
        spans, cats, descs = extract_fields(errors)

        new_record = {
            "Sentence_ID":         sid,
            "Incorrect Sentences": sentence,
            "Model_has_errors":    result.get("has_errors", ""),
            "Model_error_spans":   spans,
            "Model_categories":    cats,
            "Model_descriptions":  descs,
            "Model_corrected":     result.get("corrected_sentence", ""),
            "Status":              result.get("status", "")
        }

        # Update in-place if row existed (was failed), else append
        if sid in sid_to_idx:
            records[sid_to_idx[sid]] = new_record
        else:
            sid_to_idx[sid] = len(records)
            records.append(new_record)

        print(f"\n[{sid}] {sentence}")
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("-" * 60)

        # Save after every successful response so progress is never lost
        out_df = pd.DataFrame(records)
        out_df.to_excel(output_path, index=False)
        time.sleep(1.5)

    all_records[model_id] = records

    success = sum(1 for r in records if r.get("Status") == "success")
    failed  = sum(1 for r in records if r.get("Status") in failed_statuses)
    logger.info(f"[{model_id}] Done — {success} success, {failed} failed, "
                f"{len(records)} total saved to {output_path}")

In [ ]:
# @title
# ============================================================================
# CELL 4 — Retry Failed Rows (All Models) — loops until all resolved
# ============================================================================
failed_statuses = {"api_error", "parse_failed", "exception"}

for model_id, records in all_records.items():

    round_num  = 1
    skip_model = False

    while True:
        if skip_model:
            break

        failed_idx = [i for i, r in enumerate(records) if r.get("Status") in failed_statuses]

        if not failed_idx:
            print(f"[{model_id}] All rows resolved.")
            break

        print(f"\n[{model_id}] Round {round_num}: Retrying {len(failed_idx)} failed row(s)...\n")
        time.sleep(5)

        for i in failed_idx:
            if skip_model:
                break

            sid      = records[i]["Sentence_ID"]
            sentence = records[i]["Incorrect Sentences"]

            for attempt in range(2):
                result = call_groq(sentence, sid, model_id)

                if result.get("status") == "exception":
                    err = result.get("error", "")

                    if "404" in err or "not found" in err.lower():
                        logger.warning(f"[{model_id}] Model not found. Stopping all retries for this model.")
                        skip_model = True
                        break

                    if "429" in err or "rate limit" in err.lower():
                        if _is_daily_limit(err):
                            logger.warning(f"[{model_id}] Daily token limit hit. Stopping — retry tomorrow.")
                            skip_model = True
                            break
                        if attempt == 0:
                            wait = _parse_retry_delay(err)
                            logger.warning(f"[{sid}][{model_id}] Rate limited. Waiting {wait:.0f}s then retrying...")
                            time.sleep(wait)
                            continue

                break

            if skip_model:
                break

            errors = result.get("errors", [])
            spans, cats, descs = extract_fields(errors)

            records[i].update({
                "Sentence_ID":         sid,
                "Incorrect Sentences": sentence,
                "Model_has_errors":    result.get("has_errors", ""),
                "Model_error_spans":   spans,
                "Model_categories":    cats,
                "Model_descriptions":  descs,
                "Model_corrected":     result.get("corrected_sentence", ""),
                "Status":              result.get("status", "")
            })

            print(f"\n[{sid}] {sentence}")
            print(json.dumps(result, indent=2, ensure_ascii=False))
            print("-" * 60)

            # Save after every row so progress is never lost
            out_df = pd.DataFrame(records)
            out_df.to_excel(OUTPUT_PATHS[model_id], index=False)
            time.sleep(1.5)

        logger.info(f"[{model_id}] Round {round_num} complete. Saved to {OUTPUT_PATHS[model_id]}")
        round_num += 1

NameError: name 'all_records' is not defined